# B1 파일럿 — whisper-small 개인 LoRA (KJW, CYU)

**파이프라인이 끝까지 도는지 보는 실행이다.** 여기서 나오는 숫자는 전부 [pilot]이다. 기획서 헤드라인에 쓰지 않는다.

- 대상: KJW, CYU의 06-01 대화체 낭독. 둘 다 기준 오류가 낮은 화자다(large-v3 자모 0.056 / 0.052). 제품이 겨냥하는 사용자는 아니다.
- 문장 경계는 large-v3 단어 타임스탬프로 잡았다. **청취 검수 안 함.**
- 등록/개발/평가 분할은 학습 전에 `pilot_manifest.json`에 고정했다. 에폭 선택은 개발 세트로만 한다.

## 드라이브에 올릴 것

| 무엇 | 위치 |
| --- | --- |
| `b1_pilot` 폴더 통째로 (pilot_manifest.json, b1_train.py, cut_segments.py 등, 1 MB 미만) | 내 드라이브 최상위 |
| `b0_run.py` | 내 드라이브 최상위 — **이미 있음.** 채점 함수를 그대로 가져온다 |
| `b0b_16k` 폴더 | 내 드라이브 최상위 — **이미 있음.** 학습 조각을 여기서 잘라낸다 |

**런타임 → 런타임 유형 변경 → T4 GPU → 저장.**

## 1. 설치 — `설치 OK`가 찍혀야 다음으로 간다

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 — 런타임 유형을 T4 GPU로" ; pip -q install peft faster-whisper rapidfuzz ; pip -q uninstall -y torchao ; python -c "import torch, transformers, peft, faster_whisper; import importlib.util as u; print('설치 OK', torch.__version__, transformers.__version__, peft.__version__, 'torchao 없음' if u.find_spec('torchao') is None else 'TORCHAO 남아있음 — 런타임 재시작 후 다시')"


## 2. 드라이브 연결

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

## 3. 학습 조각 자르기 (1분 이내)

경계와 분할은 이미 `pilot_manifest.json`에 고정돼 있다. 여기서는 `b0b_16k`에서 그 구간을 잘라내기만 한다. 끝에 `cut 769 segments`와 화자별 개수가 찍혀야 한다.

In [ ]:
!cd /content/drive/MyDrive/b1_pilot && python cut_segments.py --wavdir /content/drive/MyDrive/b0b_16k

## 4. 실행 — 화자 둘, 등록 문장 전부 (약 30분)

화자마다 순서대로: 기준 small 인식(B0) → LoRA 학습(에폭마다 개발 세트 채점) → 개발 세트 기준 최선 에폭으로 평가 세트 1회 → large-v3 참고 인식.
탭을 닫지 말 것. 자동 재개는 없다.

In [ ]:
!cd /content/drive/MyDrive/b1_pilot && python b1_train.py --speaker KJW --with-large


### 4b. 둘째 화자 — 반드시 **앞 셀이 끝난 뒤** 새 셀로 돌린다

이전엔 `&&`로 묶여 있어서 KJW가 죽으면 CYU가 아예 시작도 못 했다. 자동 재개가 없으니 따로 돌린다.


In [ ]:
!cd /content/drive/MyDrive/b1_pilot && python b1_train.py --speaker CYU --with-large


## 5. (시간 남으면) 등록 30문장만으로

실제 서비스에서 요구할 수 있는 양에 가깝다. 4번과 같은 개발/평가 세트를 쓴다.

In [ ]:
!cd /content/drive/MyDrive/b1_pilot && python b1_train.py --speaker KJW --n 30 && python b1_train.py --speaker CYU --n 30

## 끝나면

`b1_pilot/results/` 아래 실행별 폴더(`KJW_nall_s0` 등)를 **통째로** 내려받아 `sw_challenge/experiments/b1_pilot/results/`에 넣는다.

- `summary.json` — 요약, 버전, 설정
- `per_segment.csv` — 문장별 짝 비교
- `hyps.json` — 원 인식 결과
- `adapter/` — 시연 서버가 읽는 개인 어댑터. `sw_challenge/demo/adapters/<실행이름>/`으로도 복사한다

`adapter_last_epoch_NOT_SELECTED/`가 생겼다면 개발 세트에서 어떤 에폭도 기준 모델을 못 이긴 것이다. **그 어댑터는 시연에 쓰지 않는다.** 그것도 결과다.

### 안 될 때

| 증상 | 조치 |
| --- | --- |
| `b0_run.py not found` | 2026-09-18부터 `b0_run.py`가 `b1_pilot` 안에 있다. 폴더를 통째로 올렸는지 확인 |
| `missing source ... b0b_16k` | `b0b_16k` 폴더가 내 드라이브 최상위에 있는지 확인 |
| `no GPU` | 런타임 유형을 T4로. 무료 티어에서 GPU가 안 잡히면 나중에 다시 |
| `large-v3 reference skipped` | 참고값만 빠진다. B0/B1 비교는 유효 |
| CUDA out of memory | 4번 명령에 `--bs 4` 추가 |
| `ImportError: Found an incompatible version of torchao` | Colab에 torchao 0.10.0이 깔려 있어 peft가 거부한다. 설치 셀이 지우도록 고쳤다 — `torchao 없음`이 찍혔는지 확인 |
| `RUNAWAY small_b0: ...` | 버그 아니다. 반복 루프가 난 세그먼트를 알려주는 것이다. 그 split의 pooled CER은 무시하고 `*_no_runaway`를 본다 |